<a href="https://colab.research.google.com/github/anastasiakalyashova/python-ai-AnastasiaKalyashova/blob/main/week3g_latitude_profile.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ═══════════════════════════════════════════════════════
#  ЯЧЕЙКА 0. Подготовка данных (из week2b_read_csv.ipynb)
#  Запускать первой в каждом ноутбуке задания 3
# ═══════════════════════════════════════════════════════

# --- Параметры (изменять здесь) ----------------------
RADIUS_KM     = 300   # радиус соседства гор (для week3a)
TOP_N_ROCKS   = 10    # сколько топ-пород использовать
TOP_N_COMPLEX = 20    # сколько самых «сложных» гор брать
# -----------------------------------------------------

import os, pandas as pd, numpy as np
from itertools import combinations

# 1. Клонируем репозиторий (если ещё нет)
repo = "python-ai-AnastasiaKalyashova"
repo_path = f"/content/{repo}"
if not os.path.exists(repo_path):
    !git clone -q https://github.com/anastasiakalyashova/python-ai-AnastasiaKalyashova.git
if os.getcwd() != repo_path:
    %cd {repo_path}

# 2. Читаем CSV
file_path = None
for root, dirs, files in os.walk("."):
    if "mountains.csv" in files:
        file_path = os.path.join(root, "mountains.csv")
        break
df = pd.read_csv(file_path)

# 3. Переименование столбцов
if "mountainLabel" in df.columns:
    df = df.rename(columns={
        "mountain":          "URL",
        "mountainLabel":     "mountain",
        "rockMaterialLabel": "rockMaterial",
        "elevationMeters":   "elevation",
    })

# 4. Нормализуем породы
df["rockMaterial"] = df["rockMaterial"].str.lower().str.strip()

# 🔧 ИСПРАВЛЕНИЕ: заменяем "lutite" на "пелит"
df["rockMaterial"] = df["rockMaterial"].replace("lutite", "пелит")

# 5. Парсим координаты
coords = df["coordinates"].str.extract(r'Point\(([^\s]+)\s+([^\s]+)\)')
df["lon"] = pd.to_numeric(coords[0], errors="coerce")
df["lat"] = pd.to_numeric(coords[1], errors="coerce")

# 6. df_unique — по одной строке на гору
df_unique = (
    df.groupby("URL")
    .agg(
        mountain   = ("mountain",     "first"),
        lon        = ("lon",          "first"),
        lat        = ("lat",          "first"),
        elevation  = ("elevation",    "first"),
        rock_count = ("rockMaterial", "nunique"),
        rocks      = ("rockMaterial", lambda x: list(x.unique())),
    )
    .reset_index()
)

# 7. df_clean — только физически возможные высоты
df_clean = df_unique[
    (df_unique.elevation >= 0) &
    (df_unique.elevation <= 8849)
].copy()

# 8. Топ пород по частоте (по df_clean)
top_rocks = (
    df[df["URL"].isin(df_clean["URL"])]
    ["rockMaterial"].value_counts()
    .head(TOP_N_ROCKS).index.tolist()
)

# 9. Co-occurrence матрица пород
pairs = []
for rocks in df_clean["rocks"]:
    clean = [r for r in rocks if r in top_rocks]
    pairs += list(combinations(sorted(set(clean)), 2))
cooc = (pd.DataFrame(pairs, columns=["r1", "r2"])
        .value_counts()
        .reset_index(name="count"))

print(f"✅ Длинный формат:    {len(df)} строк")
print(f"✅ Уникальных гор:    {len(df_unique)}")
print(f"✅ df_clean:          {len(df_clean)} гор (0–8849 м)")
print(f"✅ Топ-{TOP_N_ROCKS} пород:    {top_rocks}")
print(f"✅ Пар co-occurrence: {len(cooc)}")

/content/python-ai-AnastasiaKalyashova
✅ Длинный формат:    4431 строк
✅ Уникальных гор:    2915
✅ df_clean:          2914 гор (0–8849 м)
✅ Топ-10 пород:    ['известняк', 'песчаник', 'гранит', 'мергель', 'конгломерат', 'пелит', 'доломит', 'андезит', 'осадочная горная порода', 'базальт']
✅ Пар co-occurrence: 15


In [ ]:
# ═══════════════════════════════════════════════════════
# week3g_latitude_profile.ipynb — Земля в разрезе
# Широтный профиль гор: высота vs широта, цвет = порода
# ═══════════════════════════════════════════════════════

# 1. Загрузка подготовленных данных из предыдущей ячейки
# (предполагается, что ячейка 0 уже выполнена)

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np

# 2. Подготовка цветовой маркировки пород
# Берём топ-6 пород + "другое"
TOP_ROCKS_VIS = 6
top_rocks_vis = top_rocks[:TOP_ROCKS_VIS]  # из ячейки 0
print(f"🎨 Топ-{TOP_ROCKS_VIS} пород для визуализации: {top_rocks_vis}")

# Создаём колонку с категорией породы для окраски
def classify_rock(rocks_list):
    """Определяет основной тип породы для горы"""
    if not rocks_list:
        return "другое"
    # Берём первую породу из топ-списка, если есть
    for rock in top_rocks_vis:
        if rock in rocks_list:
            return rock
    return "другое"

df_clean['rock_main'] = df_clean['rocks'].apply(classify_rock)

# Проверяем распределение
print("\n📊 Распределение пород после классификации:")
print(df_clean['rock_main'].value_counts())

# 3. Создаём основной scatter plot
fig = px.scatter(
    df_clean,
    x='lat',
    y='elevation',
    color='rock_main',
    size='rock_count',
    hover_data={
        'mountain': True,
        'elevation': ':.0f',
        'lat': ':.2f',
        'rock_count': True,
        'rocks': True
    },
    title='🌍 Широтный профиль гор: высота vs широта',
    labels={
        'lat': 'Широта (градусы)',
        'elevation': 'Высота (м)',
        'rock_main': 'Тип породы',
        'rock_count': 'Кол-во пород'
    },
    color_discrete_sequence=px.colors.qualitative.Set2,
    size_max=15
)

# 4. Добавляем вертикальные полосы для горных систем
mountain_systems = [
    {'name': '🏔️ Гималаи', 'lat_min': 25, 'lat_max': 35,
     'y_min': 0, 'y_max': 9000, 'color': 'rgba(255, 100, 100, 0.2)'},
    {'name': '⛰️ Альпы', 'lat_min': 44, 'lat_max': 48,
     'y_min': 0, 'y_max': 5000, 'color': 'rgba(100, 255, 100, 0.2)'},
    {'name': '🗻 Анды', 'lat_min': -55, 'lat_max': -10,
     'y_min': 0, 'y_max': 7000, 'color': 'rgba(100, 100, 255, 0.2)'},
    {'name': '🏔️ Кавказ', 'lat_min': 41, 'lat_max': 43,
     'y_min': 0, 'y_max': 6000, 'color': 'rgba(255, 255, 100, 0.2)'},
    {'name': '🗻 Скалистые горы', 'lat_min': 35, 'lat_max': 60,
     'y_min': 0, 'y_max': 4500, 'color': 'rgba(255, 165, 100, 0.2)'}
]

for system in mountain_systems:
    fig.add_vrect(
        x0=system['lat_min'],
        x1=system['lat_max'],
        fillcolor=system['color'],
        opacity=0.5,
        layer='below',
        line_width=0,
        annotation_text=system['name'],
        annotation_position="top left",
        annotation_font_size=10,
        annotation_font_color="black"
    )

# 5. Добавляем горизонтальную линию высокогорья (4000 м)
fig.add_hline(
    y=4000,
    line_dash="dash",
    line_color="red",
    opacity=0.7,
    annotation_text="🏔️ высокогорье (4000 м)",
    annotation_position="bottom right",
    annotation_font_size=11,
    annotation_font_color="red"
)

# 6. Добавляем экватор для ориентира
fig.add_hline(
    y=0,
    line_dash="dot",
    line_color="gray",
    opacity=0.3,
    annotation_text="экватор",
    annotation_position="bottom left",
    annotation_font_size=9
)

# 7. Настройка осей и сетки
fig.update_layout(
    height=700,
    width=1100,
    plot_bgcolor='rgba(240, 240, 240, 0.5)',
    hovermode='closest',
    legend_title_text='Тип породы',
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01,
        bgcolor='rgba(255, 255, 255, 0.8)'
    )
)

# Улучшаем внешний вид осей
fig.update_xaxes(
    title_text="Широта (градусы)",
    gridcolor='lightgray',
    zeroline=True,
    zerolinecolor='gray',
    zerolinewidth=1
)
fig.update_yaxes(
    title_text="Высота над уровнем моря (м)",
    gridcolor='lightgray',
    zeroline=True,
    zerolinecolor='gray',
    zerolinewidth=1
)

# 8. Показываем график
fig.show()

# 9. Сохраняем в HTML (опционально)
fig.write_html("latitude_profile.html")
print("\n💾 График сохранён как 'latitude_profile.html'")

# 10. Аналитическая часть: где больше всего высокогорья?
print("\n" + "="*70)
print("📊 АНАЛИЗ: Горы выше 4000 метров по широтным поясам")
print("="*70)

# Создаём широтные зоны для анализа
lat_zones = [
    ("Южное полушарие (>30°S)", -90, -30),
    ("Умеренные широты Южн. (30°S-10°S)", -30, -10),
    ("Тропики Южн. (10°S-0°)", -10, 0),
    ("Тропики Сев. (0°-10°N)", 0, 10),
    ("Умеренные широты Сев. (10°N-30°N)", 10, 30),
    ("Субтропики/Умеренные (30°N-45°N)", 30, 45),
    ("Умеренные/Субарктика (45°N-60°N)", 45, 60),
    ("Арктика (>60°N)", 60, 90)
]

high_mountains = df_clean[df_clean['elevation'] > 4000]

print(f"\n🏔️ Всего гор выше 4000 м: {len(high_mountains)}")
print(f"   (из {len(df_clean)} гор в датасете)\n")

for zone_name, lat_min, lat_max in lat_zones:
    zone_mountains = high_mountains[
        (high_mountains['lat'] >= lat_min) &
        (high_mountains['lat'] < lat_max)
    ]
    count = len(zone_mountains)
    pct = (count / len(high_mountains)) * 100 if len(high_mountains) > 0 else 0
    bar = "█" * int(pct / 2)
    print(f"{zone_name:35} | {count:3} гор | {pct:5.1f}% {bar}")

# 11. Топ-5 широт с наибольшей концентрацией высокогорья
print("\n" + "="*70)
print("📍 ТОП-5 ШИРОТНЫХ ПОЯСОВ ПО КОНЦЕНТРАЦИИ ВЫСОКОГОРЬЯ")
print("="*70)

# Группируем по широтным диапазонам (5 градусов)
df_clean['lat_bin'] = pd.cut(df_clean['lat'], bins=np.arange(-60, 65, 5))
high_by_bin = df_clean[df_clean['elevation'] > 4000].groupby('lat_bin').size()
high_by_bin_sorted = high_by_bin.sort_values(ascending=False)

for i, (lat_range, count) in enumerate(high_by_bin_sorted.head(5).items(), 1):
    print(f"{i}. {lat_range}: {count} гор(ы)")